# 02 — Mamba SSM v1 (Google Colab / A100)

**Run on Colab:** `Runtime → Change runtime type → A100` (or T4 with reduced settings).

**Single file to upload:** `BTCUSDT_1h_unified.parquet` (~145 MB).
Place it in Google Drive or drag it into `/content/`.

**What this does:**
- Trains a Mamba state-space sequence classifier to produce P(Up) per bar.
- Walk-forward retraining every 6 months (expanding window).
- Same ATR-bracket backtester as the LGBM model → directly comparable results.
- Outputs standard artifacts: `oos_probs.npy`, `oos_index.npy`, `model_lastfold.pt`, `results.json`, PNGs.

**After Colab run:** copy `artifacts_mamba/` into `artifacts/notebooks_v2/02_mamba/` in the repo, then run the meta-learning notebook.


## 0 · Environment

In [ ]:

# Colab already ships torch, sklearn, matplotlib.
# lightgbm only needed if RUN_SELECTION=True (Boruta).
!pip -q install lightgbm 2>/dev/null
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:

import json, math, time, warnings, zipfile, calendar
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
mpl.rcParams.update({
    'font.family': 'serif', 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
})
ACCENT='#F7931A'; BLUE='#2962FF'; GREY='#9E9E9E'; RED='#EF5350'; GREEN='#26A69A'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


## 1 · Configuration

In [ ]:

# ── Data source ────────────────────────────────────────────────────────────────
USE_DRIVE  = False
DRIVE_DIR  = '/content/drive/MyDrive/hmats_data'
LOCAL_DIR  = '/content'
ARTS_DIR   = Path('/content/artifacts_mamba')
ARTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Walk-forward ───────────────────────────────────────────────────────────────
GRID_VAL_START   = pd.Timestamp('2022-01-01')
GRID_VAL_END     = pd.Timestamp('2023-12-31')
OOS_START        = pd.Timestamp('2024-01-01')
RETRAIN_MONTHS   = 6     # semi-annual (use 12 on T4 to save time)
VAL_FRAC         = 0.15
EMBARGO_H        = 24

# ── Sequence / Mamba ───────────────────────────────────────────────────────────
SEQ_LEN  = 24
STRIDE   = 1       # set to 2 on T4 to halve training time
BATCH    = 1024    # set to 512 on T4
EPOCHS   = 40
PATIENCE = 8
LR       = 3e-4
FOCAL_GAMMA = 2.0

D_MODEL=64; D_STATE=16; D_CONV=4; EXPAND=2; N_LAYERS=2; SCAN_CHUNK=8; DROPOUT=0.1

# ── Fee model (identical to LGBM) ─────────────────────────────────────────────
MAKER_FEE=0.0000; SPOT_TAKER_FEE=0.0005; FUTURES_TAKER_FEE=0.0005
BUFFER=0.0005; SHORT_FUNDING_H=0.0000077

# ── Label ─────────────────────────────────────────────────────────────────────
LABEL_COL = 'label'

# ── Feature selection ──────────────────────────────────────────────────────────
RUN_SELECTION = False   # True → re-run Boruta (~10 min CPU); False → use baked set

# Boruta-against-TBM validated feature set (from v4 Boruta run)
SELECTED_FEATURES = [
    'ret_48h','ret_72h','ret_168h','close_vs_ema_50','macd_hist_12_26','obv_z_72',
    'candles_since_cross','ma_ribbon_width','cloud_bullish','supertrend_dist_15',
    'supertrend_dir_30','fib_nearest_dist_48h','fib_dist_618_48h','fib_nearest_dist_168h',
    'close_vs_sma_504','close_vs_sma_2160','weekly_mom_accel','halving_cycle_cos',
    'halving_cycle_pos','dom_sin','quarter_cos','obv_divergence','mfi_14','cmf_20',
    'ad_z_168h','dist_round_1000','dist_round_10000','atr_14_pct_rank','bb_squeeze_20',
    'bb_squeeze_50','hurst_168h','kurt_24h','skew_168h','kurt_168h','var_ratio_6h',
    'var_ratio_24h','avg_trade_size','adf_tstat_336h','bb_width_pct',
]

# ── Backtest grid ──────────────────────────────────────────────────────────────
TRADING_GRID = {
    'long_threshold':  [0.55, 0.58, 0.60, 0.63],
    'short_threshold': [0.30, 0.35, 0.40],
    'entry_atr_mult':  [0.3,  0.6,  1.0],
    'sl_atr_mult':     [1.5,  2.0,  2.5],
    'tp_atr_mult':     [2.0,  2.5,  3.0],
    'min_sl':          [0.010, 0.015],
    'min_hold':        [4,  8],
    'max_hold':        [24, 48],
    'cooldown':        [2,  3],
}
_GRID_KEYS   = list(TRADING_GRID)
_GRID_COMBOS = list(__import__('itertools').product(*TRADING_GRID.values()))
print(f'Grid: {len(_GRID_COMBOS):,} combos  |  Features: {len(SELECTED_FEATURES)}')
print(f'Artifacts → {ARTS_DIR}')


## 2 · Load unified parquet

In [ ]:

if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    src = Path(DRIVE_DIR)
else:
    src = Path(LOCAL_DIR)

unified = src / 'BTCUSDT_1h_unified.parquet'
if not unified.exists():
    raise FileNotFoundError(
        f'{unified} not found.\n'
        'Upload BTCUSDT_1h_unified.parquet (from data/features/) to Colab or Drive.'
    )

df = pd.read_parquet(unified)
df.index = df.index.tz_localize(None) if df.index.tz else df.index

miss = [f for f in SELECTED_FEATURES if f not in df.columns]
print('Missing features:', miss if miss else 'none ✓')
print(f'Shape: {df.shape}  ({df.index.min().date()} → {df.index.max().date()})')
print(f'Label dist: {df[LABEL_COL].value_counts().to_dict()}')


## 3 · (Optional) Feature re-selection via Boruta

In [ ]:

if RUN_SELECTION:
    import lightgbm as lgb
    from scipy.stats import spearmanr
    print('Running 4-stage Boruta selection ...')
    EXCLUDE = {'open','high','low','close','volume','label'}
    cand = [c for c in df.columns if c not in EXCLUDE
            and pd.api.types.is_numeric_dtype(df[c])]
    m = df.index < OOS_START
    X  = df.loc[m, cand].fillna(0).values.astype(np.float32)
    y  = df.loc[m, LABEL_COL].values.astype(int)
    # Stage 1: variance + Spearman collinearity
    vm = X.var(0) > 1e-6; cand=[c for c,v in zip(cand,vm) if v]; X=X[:,vm]
    cm = np.abs(np.nan_to_num(spearmanr(X)[0])); np.fill_diagonal(cm,0)
    drop=set()
    for i in range(len(cand)):
        if i in drop: continue
        for j in range(i+1,len(cand)):
            if j not in drop and cm[i,j]>0.85: drop.add(j)
    cand=[c for i,c in enumerate(cand) if i not in drop]
    X=X[:,[i for i in range(X.shape[1]) if i not in drop]]
    # Stage 2: univariate AUC
    a=np.array([max((u:=roc_auc_score(y,X[:,j])),1-u) for j in range(X.shape[1])])
    keep=a>=0.505; cand=[c for c,k in zip(cand,keep) if k]; X=X[:,keep]
    # Stage 3: Boruta
    rng=np.random.default_rng(42); hits=np.zeros(len(cand))
    for t in tqdm(range(30),desc='Boruta'):
        Xs=X.copy()
        for j in range(Xs.shape[1]): rng.shuffle(Xs[:,j])
        mdl=lgb.LGBMClassifier(n_estimators=150,num_leaves=31,learning_rate=0.05,
            subsample=0.7,colsample_bytree=0.7,min_child_samples=20,verbose=-1,
            n_jobs=-1,random_state=int(t)).fit(np.hstack([X,Xs]),y)
        imp=mdl.feature_importances_; nf=len(cand)
        hits+=(imp[:nf]>imp[nf:].max()).astype(float)
    SELECTED_FEATURES=[c for c,h in zip(cand,hits/30) if h>0.5]
    print(f'Selected {len(SELECTED_FEATURES)}:', SELECTED_FEATURES)
else:
    print(f'Using baked feature set: {len(SELECTED_FEATURES)} features')


## 4 · Mamba architecture

In [ ]:

def _chunked_ssm_scan(x, delta, A, B_mat, C_mat, D_coef, chunk=SCAN_CHUNK):
    B, L, D = x.shape; N = A.shape[-1]
    log_dA = delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)
    dBu    = delta.unsqueeze(-1) * B_mat.unsqueeze(2) * x.unsqueeze(-1)
    n_chunks = (L + chunk - 1) // chunk; L_pad = n_chunks * chunk
    if L_pad > L:
        pad    = L_pad - L
        log_dA = F.pad(log_dA,(0,0,0,0,0,pad)); dBu=F.pad(dBu,(0,0,0,0,0,pad))
        C_pad  = F.pad(C_mat,(0,0,0,pad))
    else:
        C_pad = C_mat
    log_dA_c=log_dA.view(B,n_chunks,chunk,D,N); dBu_c=dBu.view(B,n_chunks,chunk,D,N)
    C_c=C_pad.view(B,n_chunks,chunk,N)
    P_c=torch.exp(torch.cumsum(log_dA_c,dim=2))
    within_c=P_c*torch.cumsum(dBu_c/P_c.clamp(min=1e-12),dim=2)
    P_full_c=torch.exp(log_dA_c.sum(dim=2))
    y=x.new_zeros(B,L_pad,D); h=x.new_zeros(B,D,N)
    for ci in range(n_chunks):
        h_chunk=P_c[:,ci]*h.unsqueeze(1)+within_c[:,ci]
        y[:,ci*chunk:(ci+1)*chunk]=(h_chunk*C_c[:,ci].unsqueeze(-2)).sum(-1)
        h=P_full_c[:,ci]*h+within_c[:,ci,-1]
    return y[:,:L]+x*D_coef.unsqueeze(0).unsqueeze(0)

class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND):
        super().__init__()
        d_inner=d_model*expand; dt_rank=max(1,d_model//16)
        self.d_inner=d_inner; self.d_state=d_state; self.dt_rank=dt_rank
        A_init=torch.arange(1,d_state+1,dtype=torch.float32).repeat(d_inner,1)
        self.A_log=nn.Parameter(torch.log(A_init)); self.D=nn.Parameter(torch.ones(d_inner))
        self.norm=nn.LayerNorm(d_model); self.in_proj=nn.Linear(d_model,d_inner*2,bias=False)
        self.conv1d=nn.Conv1d(d_inner,d_inner,d_conv,padding=d_conv-1,groups=d_inner,bias=True)
        self.x_proj=nn.Linear(d_inner,dt_rank+d_state*2,bias=False)
        self.dt_proj=nn.Linear(dt_rank,d_inner,bias=True)
        nn.init.constant_(self.dt_proj.bias,math.log(math.expm1(1.0)))
        self.out_proj=nn.Linear(d_inner,d_model,bias=False)
    def forward(self,x):
        res=x; x=self.norm(x); L=x.shape[1]
        x_in,z=self.in_proj(x).chunk(2,dim=-1)
        x_in=F.silu(self.conv1d(x_in.transpose(1,2))[...,:L].transpose(1,2))
        dt,B_ssm,C=self.x_proj(x_in).split([self.dt_rank,self.d_state,self.d_state],dim=-1)
        delta=F.softplus(self.dt_proj(dt)); A=-torch.exp(self.A_log.float())
        y=_chunked_ssm_scan(x_in,delta,A,B_ssm,C,self.D)
        return self.out_proj(y*F.silu(z))+res

class MambaClassifier(nn.Module):
    def __init__(self, n_features, d_model=D_MODEL, n_layers=N_LAYERS,
                 d_state=D_STATE, n_classes=2, dropout=DROPOUT):
        super().__init__()
        self.input_proj=nn.Linear(n_features,d_model)
        self.blocks=nn.ModuleList([MambaBlock(d_model,d_state) for _ in range(n_layers)])
        self.dropout=nn.Dropout(dropout); self.head=nn.Linear(d_model,n_classes)
    def forward(self,x):
        x=self.input_proj(x)
        for blk in self.blocks: x=blk(x)
        return self.head(self.dropout(x[:,-1,:]))

class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,alpha=None):
        super().__init__(); self.gamma=gamma; self.register_buffer('alpha',alpha)
    def forward(self,logits,targets):
        ce=F.cross_entropy(logits,targets,reduction='none',weight=self.alpha)
        return ((1-torch.exp(-ce))**self.gamma*ce).mean()

_m=MambaClassifier(n_features=len(SELECTED_FEATURES)).to(DEVICE)
print(f'MambaClassifier: {sum(p.numel() for p in _m.parameters()):,} params  |  '
      f'd_model={D_MODEL} d_state={D_STATE} layers={N_LAYERS}')
del _m


## 5 · Walk-forward training

In [ ]:

FEATS  = SELECTED_FEATURES
F_IDX  = {c:i for i,c in enumerate(FEATS)}
y_all  = df[LABEL_COL].values.astype(float)

def build_sequences(X, y, seq_len, stride=1):
    ends  = np.arange(seq_len-1,len(X),stride)
    valid = ~np.isnan(y[ends].astype(float))
    ends  = ends[valid]
    return np.stack([X[e-seq_len+1:e+1] for e in ends],0).astype(np.float32), y[ends].astype(np.int64)

def make_focal(y_arr):
    counts=np.bincount(y_arr.astype(int),minlength=2)
    alpha=torch.tensor(1.0/(counts+1),dtype=torch.float32,device=DEVICE)
    return FocalLoss(FOCAL_GAMMA,alpha/alpha.sum())

def train_fold(X_tr_s,y_tr_s,X_vl_s,y_vl_s,n_feat):
    ld_tr=DataLoader(TensorDataset(torch.from_numpy(X_tr_s),torch.from_numpy(y_tr_s)),
                     BATCH,shuffle=True,drop_last=True)
    ld_vl=DataLoader(TensorDataset(torch.from_numpy(X_vl_s),torch.from_numpy(y_vl_s)),
                     BATCH,shuffle=False)
    model=MambaClassifier(n_features=n_feat).to(DEVICE)
    crit=make_focal(y_tr_s)
    opt=AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    sched=SequentialLR(opt,[LinearLR(opt,0.1,1.0,total_iters=5),
        CosineAnnealingLR(opt,T_max=EPOCHS-5,eta_min=1e-6)],milestones=[5])
    best_auc=0.; best_state=None; best_ep=0
    for ep in range(1,EPOCHS+1):
        model.train()
        for xb,yb in ld_tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            loss=crit(model(xb),yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        sched.step()
        model.eval(); ps=[]; ys=[]
        with torch.no_grad():
            for xb,yb in ld_vl:
                ps.append(torch.softmax(model(xb.to(DEVICE)),-1)[:,1].cpu().numpy())
                ys.append(yb.numpy())
        p=np.concatenate(ps); yv=np.concatenate(ys)
        auc=roc_auc_score(yv,p) if 0<yv.sum()<len(yv) else 0.5
        if auc>best_auc: best_auc=auc; best_ep=ep; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        if ep-best_ep>=PATIENCE: break
    model.load_state_dict(best_state)
    return model, best_auc, best_ep

# Walk-forward loop
probs    = np.full(len(df), np.nan)
X_raw    = np.nan_to_num(df[FEATS].values.astype(np.float32))
anchors  = []; d = pd.Timestamp('2022-01-01')
while d <= df.index[-1]: anchors.append(d); d += pd.DateOffset(months=RETRAIN_MONTHS)

best_model=None; last_qt=None; fold_log=[]; t_all=time.time()
for fi, a_start in enumerate(anchors):
    a_end    = a_start + pd.DateOffset(months=RETRAIN_MONTHS)
    tr_cut   = a_start - pd.Timedelta(hours=EMBARGO_H)
    tr_mask  = df.index < tr_cut
    if tr_mask.sum() < 5000: continue
    tr_idx   = np.where(tr_mask)[0]
    n_val    = int(len(tr_idx)*VAL_FRAC)
    tr_rows, vl_rows = tr_idx[:-n_val], tr_idx[-n_val:]
    qt       = QuantileTransformer(n_quantiles=1000,output_distribution='normal',random_state=42)
    qt.fit(X_raw[tr_rows])
    X_scaled = qt.transform(X_raw)
    X_tr_s,y_tr_s = build_sequences(X_scaled[tr_rows[0]:tr_rows[-1]+1],
                                    y_all[tr_rows[0]:tr_rows[-1]+1], SEQ_LEN, STRIDE)
    X_vl_s,y_vl_s = build_sequences(X_scaled[vl_rows[0]-SEQ_LEN+1:vl_rows[-1]+1],
                                    y_all[vl_rows[0]-SEQ_LEN+1:vl_rows[-1]+1], SEQ_LEN, 1)
    model,auc,ep = train_fold(X_tr_s,y_tr_s,X_vl_s,y_vl_s,len(FEATS))
    oos_mask = (df.index>=a_start)&(df.index<a_end)
    oos_rows = np.where(oos_mask)[0]; oos_rows=oos_rows[oos_rows>=SEQ_LEN-1]
    if len(oos_rows):
        Xo=np.stack([X_scaled[e-SEQ_LEN+1:e+1] for e in oos_rows]).astype(np.float32)
        model.eval(); preds=[]
        with torch.no_grad():
            for ch in torch.split(torch.from_numpy(Xo),2048):
                preds.append(torch.softmax(model(ch.to(DEVICE)),-1)[:,1].cpu().numpy())
        probs[oos_rows]=np.concatenate(preds)
    best_model=model; last_qt=qt
    fold_log.append({'fold':fi,'val_auc':round(auc,4),'ep':ep,
                     'oos':f'{a_start.date()}→{a_end.date()}'})
    print(f'  fold {fi}: <{tr_cut.date()}  OOS {a_start.date()}→{a_end.date()}  auc={auc:.4f}@{ep}  ({len(oos_rows)} bars)')

print(f'WFO done in {(time.time()-t_all)/60:.1f} min  |  {len(fold_log)} folds')
torch.save(best_model.state_dict(), ARTS_DIR/'model_lastfold.pt')

# OOS metrics
oos_m = (df.index>=OOS_START)&(~np.isnan(probs))
auc_oos=roc_auc_score(y_all[oos_m],probs[oos_m])
oos_probs_series=pd.Series(probs,index=df.index)[df.index>=OOS_START]
oos_df = df[df.index>=OOS_START].copy()
print(f'OOS AUC: {auc_oos:.4f}  bars={oos_m.sum():,}')


## 6 · Grid search (2022–2023 validation window)

In [ ]:

def _run_backtest(probs_arr, close_arr, high_arr, low_arr, atr_arr,
        long_threshold, short_threshold, entry_atr_mult, sl_atr_mult, tp_atr_mult,
        min_sl, min_hold, max_hold, cooldown, with_fees=True):
    n=len(close_arr); eq=np.ones(n); cur=1.0; trades=[]
    in_pos=False; direction=None; entry_px=sl_px=tp_px=pos_eq=entry_fee=0.0
    hold_cnt=cd_cnt=0; funding=0.0; pending=None
    for i in range(n):
        lo=low_arr[i]; hi=high_arr[i]; px=close_arr[i]
        if in_pos:
            hold_cnt+=1
            if direction=='short': funding+=SHORT_FUNDING_H
            eq[i]=pos_eq*(px/entry_px if direction=='long' else 1+(entry_px-px)/entry_px)
            exited=False; exit_px=0.; reason=''; exit_fee=0.
            if hold_cnt>=min_hold:
                if direction=='long':
                    if lo<=sl_px: exit_px=sl_px;exited=True;reason='sl';exit_fee=SPOT_TAKER_FEE if with_fees else 0.
                    elif hi>=tp_px: exit_px=tp_px;exited=True;reason='tp';exit_fee=MAKER_FEE
                    elif hold_cnt>=max_hold: exit_px=px;exited=True;reason='timeout';exit_fee=SPOT_TAKER_FEE if with_fees else 0.
                else:
                    if hi>=sl_px: exit_px=sl_px;exited=True;reason='sl';exit_fee=FUTURES_TAKER_FEE if with_fees else 0.
                    elif lo<=tp_px: exit_px=tp_px;exited=True;reason='tp';exit_fee=MAKER_FEE
                    elif hold_cnt>=max_hold: exit_px=px;exited=True;reason='timeout';exit_fee=FUTURES_TAKER_FEE if with_fees else 0.
            if exited:
                gross=((exit_px-entry_px)/entry_px if direction=='long' else (entry_px-exit_px)/entry_px)
                net=gross-(entry_fee+exit_fee if with_fees else 0.)+funding
                cur=pos_eq*(1.+net); eq[i]=cur
                trades.append({'direction':direction,'reason':reason,'gross':gross,'net':net,'hold':hold_cnt})
                in_pos=False; cd_cnt=cooldown; funding=0.
        elif pending is not None:
            d,lim,p_sl,p_tp=pending
            if d=='long': filled=lo<=lim+BUFFER; ef=MAKER_FEE if (filled and with_fees) else (SPOT_TAKER_FEE if with_fees else 0.)
            else: filled=hi>=lim-BUFFER; ef=MAKER_FEE if (filled and with_fees) else (FUTURES_TAKER_FEE if with_fees else 0.)
            entry_px=lim if filled else px; sl_px=p_sl; tp_px=p_tp; entry_fee=ef
            direction=d; in_pos=True; pos_eq=cur; hold_cnt=0; funding=0.; pending=None; eq[i]=cur
        elif cd_cnt>0: cd_cnt-=1; eq[i]=cur
        elif not np.isnan(probs_arr[i]) and i+1<n:
            atr=max(atr_arr[i],min_sl)
            if probs_arr[i]>long_threshold:
                pending=('long',px*(1-entry_atr_mult*atr),px*(1-sl_atr_mult*atr),px*(1+tp_atr_mult*atr))
            elif probs_arr[i]<short_threshold:
                pending=('short',px*(1+entry_atr_mult*atr),px*(1+sl_atr_mult*atr),px*(1-tp_atr_mult*atr))
            eq[i]=cur
        else: eq[i]=cur
    if in_pos:
        gross=((px-entry_px)/entry_px if direction=='long' else (entry_px-px)/entry_px)
        taker=SPOT_TAKER_FEE if direction=='long' else FUTURES_TAKER_FEE
        net=gross-(entry_fee+(taker if with_fees else 0.))+funding; cur=pos_eq*(1.+net); eq[-1]=cur
    return eq, trades

def _sharpe(eq):
    r=np.diff(np.log(np.maximum(eq,1e-12))); return float(r.mean()/(r.std(ddof=1)+1e-12)*np.sqrt(24*365))
def _maxdd(eq):
    pk=np.maximum.accumulate(eq); return float(((eq-pk)/(pk+1e-12)).min())

gv_m=(df.index>=GRID_VAL_START)&(df.index<=GRID_VAL_END); gv_df=df[gv_m]
gv_p=probs[gv_m.values]
rows=[]; t0=time.time()
for vals in _GRID_COMBOS:
    p=dict(zip(_GRID_KEYS,vals))
    if p['short_threshold']>=p['long_threshold'] or p['max_hold']<p['min_hold']: continue
    eq,tr=_run_backtest(gv_p,gv_df['close'].values,gv_df['high'].values,
                        gv_df['low'].values,gv_df['atr_14_pct'].values,with_fees=True,**p)
    if len(tr)<15: continue
    rows.append({**p,'sharpe':_sharpe(eq),'total_ret':float(eq[-1]-1),
                 'maxdd':_maxdd(eq),'n_trades':len(tr)})
grid_df=pd.DataFrame(rows).sort_values('sharpe',ascending=False).reset_index(drop=True)
INT={'min_hold','max_hold','cooldown'}
BEST={k:(int(grid_df.iloc[0][k]) if k in INT else float(grid_df.iloc[0][k])) for k in _GRID_KEYS}
print(f'Grid done in {time.time()-t0:.0f}s  |  {len(grid_df):,} valid combos')
print('Best params:'); [print(f'  {k:<18}{v}') for k,v in BEST.items()]


## 7 · OOS backtest & figures

In [ ]:

o_p=oos_probs_series.values
o_idx=oos_df.index; bh=(oos_df['close'].values/oos_df['close'].iloc[0]-1)*100

eq_fees,tdf_fees=_run_backtest(o_p,oos_df['close'].values,oos_df['high'].values,
    oos_df['low'].values,oos_df['atr_14_pct'].values,with_fees=True,**BEST)
eq_0fee,tdf_0fee=_run_backtest(o_p,oos_df['close'].values,oos_df['high'].values,
    oos_df['low'].values,oos_df['atr_14_pct'].values,with_fees=False,**BEST)
TF=pd.DataFrame(tdf_fees) if tdf_fees else pd.DataFrame(columns=['direction','reason','gross','net','hold'])
T0=pd.DataFrame(tdf_0fee) if tdf_0fee else pd.DataFrame(columns=['direction','reason','gross','net','hold'])

print(f'{"":22}{"Trades":>7}{"Win":>8}{"Return":>9}{"Sharpe":>8}{"MaxDD":>8}')
print('-'*62)
for lbl,eq,t in [('With fees',eq_fees,TF),('Zero-fee',eq_0fee,T0)]:
    wr=(t['net']>0).mean() if len(t) else 0
    print(f'{lbl:22}{len(t):>7}{wr:>8.1%}{eq[-1]-1:>+9.1%}{_sharpe(eq):>8.3f}{_maxdd(eq):>8.1%}')
print(f'BTC Buy & Hold:                              {bh[-1]:>+9.1f}%')

fig,(ax1,ax2)=plt.subplots(2,1,figsize=(13,7),height_ratios=[3,1],sharex=True)
ax1.plot(o_idx,(eq_fees-1)*100,color=ACCENT,lw=1.5,label='Mamba (w/ fees)')
ax1.plot(o_idx,(eq_0fee-1)*100,color=BLUE,lw=1.0,ls='--',label='Mamba (0-fee)')
ax1.plot(o_idx,bh,color=GREY,lw=1.0,ls=':',label='BTC B&H')
ax1.axhline(0,color=GREY,lw=0.6,ls=':'); ax1.set_ylabel('Return (%)'); ax1.legend()
ax1.set_title(f'Mamba v1 — OOS {OOS_START.date()}→{o_idx[-1].date()} | '
    f'Sharpe={_sharpe(eq_fees):.2f}  Return={eq_fees[-1]-1:+.1%}  MaxDD={_maxdd(eq_fees):.1%}  AUC={auc_oos:.3f}',
    fontweight='bold')
pk=np.maximum.accumulate(eq_fees); dd=(eq_fees-pk)/pk*100
ax2.fill_between(o_idx,dd,0,color=RED,alpha=0.4); ax2.set_ylabel('DD (%)')
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
plt.setp(ax2.xaxis.get_majorticklabels(),rotation=30,ha='right')
fig.tight_layout(); fig.savefig(ARTS_DIR/'01_equity_drawdown.png'); plt.show()

eqs=pd.Series(eq_fees,index=o_idx); mret=eqs.resample('ME').last().pct_change().fillna(0)*100
fig,ax=plt.subplots(figsize=(13,4))
ax.bar(mret.index,mret.values,color=[GREEN if r>=0 else RED for r in mret],width=22,alpha=0.75)
ax.plot(mret.index,mret.rolling(3,min_periods=1).mean(),color=BLUE,lw=2,label='3-mo SMA')
ax.axhline(0,color=GREY,lw=0.8,ls=':'); ax.set_ylabel('Return (%)'); ax.legend()
ax.set_title(f'Monthly Returns — Mamba v1 | positive {int((mret>0).sum())}/{len(mret)} avg {mret.mean():+.2f}%',fontweight='bold')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha='right')
fig.tight_layout(); fig.savefig(ARTS_DIR/'02_monthly_returns.png'); plt.show()

cal=mret.to_frame('r'); cal['y']=cal.index.year; cal['m']=cal.index.month
piv=cal.pivot(index='y',columns='m',values='r'); piv.columns=[calendar.month_abbr[m] for m in piv.columns]
fig,ax=plt.subplots(figsize=(12,max(2,len(piv)*0.7)))
sns.heatmap(piv,ax=ax,cmap='RdYlGn',center=0,annot=True,fmt='.1f',linewidths=0.5)
ax.set_title('Monthly Return Calendar — Mamba v1',fontweight='bold'); ax.set_xlabel(''); ax.set_ylabel('')
fig.tight_layout(); fig.savefig(ARTS_DIR/'03_monthly_heatmap.png'); plt.show()


## 8 · Save artifacts & download

In [ ]:

np.save(ARTS_DIR/'oos_probs.npy',  oos_probs_series.values.astype(np.float32))
np.save(ARTS_DIR/'oos_index.npy',  oos_df.index.astype(np.int64).values)

def _bt_metrics(eq,t):
    wr=float((t['net']>0).mean()) if len(t) else 0.
    nl=int((t['direction']=='long').sum()) if len(t) else 0
    ns=int((t['direction']=='short').sum()) if len(t) else 0
    return {'n_trades':len(t),'n_long':nl,'n_short':ns,'win_rate':round(wr,4),
            'total_ret':round(float(eq[-1]-1),4),'sharpe':round(_sharpe(eq),4),'maxdd':round(_maxdd(eq),4)}

results={
    'notebook':'02_mamba_v1','created':pd.Timestamp.now().isoformat(),
    'model':'Mamba SSM (pure-PyTorch, chunked scan)',
    'wfo':{'retrain_months':RETRAIN_MONTHS,'folds':fold_log},
    'grid_val':f'{GRID_VAL_START.date()}→{GRID_VAL_END.date()}',
    'oos_period':f'{OOS_START.date()}→{oos_df.index[-1].date()}',
    'oos_auc':round(float(auc_oos),4),
    'architecture':{'d_model':D_MODEL,'d_state':D_STATE,'n_layers':N_LAYERS,'seq_len':SEQ_LEN,'batch':BATCH},
    'selected_features':SELECTED_FEATURES,
    'best_params':BEST,
    'backtest_wfees':_bt_metrics(eq_fees,TF),
    'backtest_0fee':_bt_metrics(eq_0fee,T0),
    'monthly':{'mean_pct':round(float(mret.mean()),3),'positive_months':int((mret>0).sum()),'total_months':int(len(mret))},
    'artifacts':{'oos_probs':'oos_probs.npy','oos_index':'oos_index.npy','model':'model_lastfold.pt'},
}
with open(ARTS_DIR/'results.json','w') as f: json.dump(results,f,indent=2,default=str)
TF.to_csv(ARTS_DIR/'trades_wfees.csv',index=False)

# Zip everything and download
zp='/content/mamba_artifacts.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ARTS_DIR.iterdir(): z.write(p,p.name)
print('Zipped →',zp)
try:
    from google.colab import files; files.download(zp)
except: pass

# Copy to repo location (only if run locally)
print('\nNow copy artifacts_mamba/ into:')
print('  <repo>/artifacts/notebooks_v2/02_mamba/')
print('Then run 04_meta_learning_v1.ipynb.')
